# Topic 2: Window Functions

> **Phase 3 → Week 4 → Topic 2**
>
> Prerequisites: Topic 1 (Joins)

---

## The Exam Ranking Analogy

Imagine a class exam. You want to:
- Rank each student within their section (not overall — within group)
- Show each student's score AND the section average in the same row
- Show how much each student improved from their last exam

With regular `groupBy`, you lose individual rows — you can only see aggregates.
**Window functions** let you compute aggregates while keeping every row.

```
Regular groupBy:                    Window Function:
Dept       Avg_Salary               Name    Dept    Salary   Dept_Avg  Rank
--------   ----------               ------  ----    ------   --------  ----
Eng        91500          vs        Alice   Eng     95000    91500       1
Sales      55000                    Bob     Eng     88000    91500       2
                                    Eve     Sales   55000    55000       1

groupBy loses individual rows!      Window keeps ALL rows with context!
```

---

## Window Specification — 3 Parts

Every window function needs a **WindowSpec** that defines:

```python
from pyspark.sql.window import Window

window = (
    Window
    .partitionBy("dept")          # 1. PARTITION: reset the window per dept (like GROUP BY)
    .orderBy("salary")            # 2. ORDER:     sort within each partition
    .rowsBetween(-2, 0)           # 3. FRAME:     which rows to include in calculation
)                                 #               (optional — ranking functions don't need it)
```

### Frame Types:
```
Window.unboundedPreceding  → from the first row of partition
Window.currentRow          → current row
Window.unboundedFollowing  → to the last row of partition

rowsBetween(start, end)  → physical rows (by position)
rangeBetween(start, end) → logical range (by value — requires ORDER BY)
```

---

## Categories of Window Functions

| Category | Functions |
|----------|----------|
| **Ranking** | `row_number()`, `rank()`, `dense_rank()`, `ntile(n)`, `percent_rank()` |
| **Analytic** | `lag(col, n)`, `lead(col, n)`, `first()`, `last()` |
| **Aggregate** | `sum()`, `avg()`, `min()`, `max()`, `count()` over a window |

---

## Interview Cheat Sheet

**Q: What is a window function in Spark?**
> A window function computes a value for each row based on a group of rows (the "window") related to that row — without collapsing rows like groupBy does. The window is defined by a partition (reset per group), an order (sort within group), and optionally a frame (range of rows). Example: ranking employees by salary within each department while keeping all rows.

**Q: Difference between rank(), dense_rank(), and row_number()?**
> Given scores [100, 100, 90]: `row_number()` → 1,2,3 (always unique, arbitrary tiebreak); `rank()` → 1,1,3 (ties get same rank, gaps after); `dense_rank()` → 1,1,2 (ties get same rank, NO gaps). Use `row_number` for deduplication (get exactly one per group), `dense_rank` for leaderboards.

**Q: How do lag() and lead() work?**
> `lag(col, n)` returns the value of `col` from n rows BEFORE the current row in the window; `lead(col, n)` returns n rows AFTER. Both require an ORDER BY in the WindowSpec. Classic use: month-over-month comparison — `lag(revenue, 1)` gives last month's revenue so you can compute `revenue - lag(revenue, 1)` as the change.

**Q: What's the difference between rowsBetween and rangeBetween?**
> `rowsBetween` counts physical rows (positions). `rangeBetween` counts by VALUE relative to the current row's ORDER BY value. For a 3-row rolling average: `rowsBetween(-2, 0)` = previous 2 physical rows. If two rows have the same ORDER BY value, `rangeBetween` includes all of them; `rowsBetween` handles them individually.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Week4 - Window Functions") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Load sales data
sales = spark.read.csv("/workspace/week4/data/sales.csv", header=True, inferSchema=True)
sales.show()
print(f"Sales data: {sales.count()} rows")

## Part 1: Ranking Functions

In [ ]:
# row_number, rank, dense_rank — all on salary within dept
employees = spark.createDataFrame([
    ("Alice",   "Eng",   95000),
    ("Bob",     "Eng",   88000),
    ("Charlie", "Eng",   88000),  # same salary as Bob — shows rank vs dense_rank difference
    ("Dave",    "Eng",   72000),
    ("Eve",     "Sales", 65000),
    ("Frank",   "Sales", 55000),
    ("Grace",   "Sales", 55000),  # tied
], ["name", "dept", "salary"])

# Define window: partition by dept, order by salary desc
w = Window.partitionBy("dept").orderBy(F.col("salary").desc())

ranked = employees.withColumn("row_number",  F.row_number().over(w)) \
                  .withColumn("rank",         F.rank().over(w)) \
                  .withColumn("dense_rank",   F.dense_rank().over(w))

print("Ranking functions comparison (ties at Bob/Charlie and Frank/Grace):")
ranked.show()

print("""
row_number: always 1,2,3,4 — no ties, arbitrary order for equal values
rank:       1,2,2,4 — ties get same rank, SKIPS next rank (gap after tie)
dense_rank: 1,2,2,3 — ties get same rank, NO gap
""")

In [ ]:
# MOST COMMON INTERVIEW PATTERN: Get top-N per group (deduplication)
# "Find the top 2 earners in each department"

top2_per_dept = ranked.filter(F.col("row_number") <= 2) \
                      .select("dept", "name", "salary", "row_number")

print("Top 2 earners per department:")
top2_per_dept.show()

# Why row_number and not rank? rank could give >2 results if 2nd place is tied
top2_rank = ranked.filter(F.col("rank") <= 2).select("dept", "name", "salary", "rank")
print("Using rank (may return more than 2 due to ties):")
top2_rank.show()

In [ ]:
# ntile(n) — divide rows into n buckets (like percentiles)
# ntile(4) = quartiles: bucket 1 = bottom 25%, bucket 4 = top 25%

w_overall = Window.orderBy(F.col("salary").desc())

quartile_df = employees.withColumn("quartile", F.ntile(4).over(w_overall)) \
                       .withColumn("pct_rank",  F.round(F.percent_rank().over(w_overall), 2))

print("ntile(4) and percent_rank:")
quartile_df.show()
print("Quartile 1 = top earners, Quartile 4 = bottom earners")

## Part 2: Analytic Functions (lag, lead)

In [ ]:
# lag() and lead() — look backward/forward in the window
# Classic use: month-over-month revenue change per salesperson

# Create ordered monthly sales per salesperson
monthly_order = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6}
month_map_expr = F.when(F.col("month") == "January", 1) \
                  .when(F.col("month") == "February", 2) \
                  .when(F.col("month") == "March", 3) \
                  .when(F.col("month") == "April", 4) \
                  .when(F.col("month") == "May", 5) \
                  .otherwise(6)

sales_ordered = sales.withColumn("month_num", month_map_expr)

# Window: per salesperson, ordered by month
w_person = Window.partitionBy("salesperson").orderBy("month_num")

lag_lead_df = sales_ordered.withColumn(
    "prev_revenue",   F.lag("revenue", 1).over(w_person)          # previous month
).withColumn(
    "next_revenue",   F.lead("revenue", 1).over(w_person)         # next month
).withColumn(
    "mom_change",     F.col("revenue") - F.col("prev_revenue")    # month-over-month
).withColumn(
    "mom_pct",        F.round((F.col("mom_change") / F.col("prev_revenue")) * 100, 1)
)

print("Month-over-month revenue analysis using lag():")
lag_lead_df.filter(F.col("salesperson") == "Alice") \
           .select("salesperson", "month", "revenue", "prev_revenue", "mom_change", "mom_pct") \
           .show()

In [ ]:
# first() and last() over a window — useful for forward/backward fill
# Example: for each sale, show the first and most recent revenue for that salesperson

w_first = Window.partitionBy("salesperson").orderBy("month_num") \
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)

w_last  = Window.partitionBy("salesperson").orderBy("month_num") \
                .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

first_last_df = sales_ordered.withColumn("running_max", F.max("revenue").over(w_first)) \
                              .withColumn("season_max",   F.max("revenue").over(w_last))

print("Running max vs full-season max for Alice:")
first_last_df.filter(F.col("salesperson") == "Alice") \
             .select("month", "revenue", "running_max", "season_max") \
             .show()

## Part 3: Aggregate Window Functions

In [ ]:
# Running total (cumulative sum) — classic interview question
# Frame: from first row of partition to current row

w_cumulative = Window.partitionBy("salesperson") \
                     .orderBy("month_num") \
                     .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total = sales_ordered.withColumn(
    "cumulative_revenue", F.sum("revenue").over(w_cumulative)
)

print("Running total (cumulative sum) per salesperson:")
running_total.filter(F.col("salesperson") == "Bob") \
             .select("salesperson", "month", "revenue", "cumulative_revenue") \
             .show()

In [ ]:
# Rolling average (moving average) — previous 2 months + current
w_rolling = Window.partitionBy("salesperson") \
                  .orderBy("month_num") \
                  .rowsBetween(-2, 0)   # 2 rows before + current

rolling_avg = sales_ordered.withColumn(
    "rolling_3m_avg", F.round(F.avg("revenue").over(w_rolling), 0)
)

print("3-month rolling average for Carol:")
rolling_avg.filter(F.col("salesperson") == "Carol") \
           .select("month", "revenue", "rolling_3m_avg") \
           .show()

In [ ]:
# Window aggregate without frame = entire partition aggregate
# "Each row shows its own value AND the dept/region total"

w_region = Window.partitionBy("region")

with_region_stats = sales.withColumn(
    "region_total",   F.sum("revenue").over(w_region)
).withColumn(
    "region_avg",     F.round(F.avg("revenue").over(w_region), 0)
).withColumn(
    "pct_of_region",  F.round(F.col("revenue") / F.col("region_total") * 100, 1)
)

print("Each sale's contribution to its region total:")
with_region_stats.select("salesperson", "region", "month", "revenue",
                          "region_total", "pct_of_region") \
                 .orderBy("region", "month") \
                 .show()

## Part 4: Real-World Patterns

In [ ]:
# PATTERN 1: Deduplication — keep only the LATEST record per key
# Classic use: CDC (change data capture) — multiple updates for same entity

cdc_data = spark.createDataFrame([
    ("user_1", "Alice",   "alice@old.com",  "2023-01-01"),
    ("user_1", "Alice",   "alice@new.com",  "2023-06-15"),   # update
    ("user_1", "Alice M", "alice@new.com",  "2023-08-20"),   # latest
    ("user_2", "Bob",     "bob@email.com",  "2023-03-10"),
    ("user_2", "Bob B",   "bob@email.com",  "2023-09-01"),   # latest
], ["user_id", "name", "email", "updated_at"])

w_dedup = Window.partitionBy("user_id").orderBy(F.col("updated_at").desc())

latest_records = cdc_data.withColumn("rn", F.row_number().over(w_dedup)) \
                          .filter(F.col("rn") == 1) \
                          .drop("rn")

print(f"Original CDC records: {cdc_data.count()}")
print(f"After dedup (latest only): {latest_records.count()}")
latest_records.show()

In [ ]:
# PATTERN 2: Gap & Island — find consecutive sequences
# "Find periods where a salesperson was continuously in top 2 by region"

# Simpler version: detect streaks of improving revenue
w_streak = Window.partitionBy("salesperson").orderBy("month_num")

streak_df = sales_ordered \
    .withColumn("prev_rev", F.lag("revenue", 1).over(w_streak)) \
    .withColumn("is_increase", (F.col("revenue") > F.col("prev_rev")).cast("int")) \
    .fillna({"is_increase": 0})

print("Month-over-month increases (1 = increased, 0 = decreased/same):")
streak_df.select("salesperson", "month", "revenue", "is_increase") \
         .orderBy("salesperson", "month_num") \
         .show()

## Exercises

1. Using the `sales` data, rank each salesperson by total revenue (all months combined). Use `dense_rank()`.
2. For each salesperson, calculate the cumulative units_sold month by month.
3. Find the month where each salesperson had their highest revenue (use `row_number` to get rank 1 within each person).
4. For each region, show the top-performing salesperson per month (hint: partition by region AND month, rank by revenue).
5. Calculate the 2-month rolling average revenue for each region.

In [ ]:
# Exercise 1: Rank salespersons by total revenue
total_rev = sales.groupBy("salesperson").agg(F.sum("revenue").alias("total_revenue"))
w_rank = Window.orderBy(F.col("total_revenue").desc())
total_rev.withColumn("rank", F.dense_rank().over(w_rank)).show()

In [ ]:
# Exercise 3: Best month per salesperson
w_best = Window.partitionBy("salesperson").orderBy(F.col("revenue").desc())
sales_ordered.withColumn("rn", F.row_number().over(w_best)) \
             .filter(F.col("rn") == 1) \
             .select("salesperson", "month", "revenue") \
             .show()